In [ ]:
import scrapy
import pandas as pd
from scrapy.crawler import CrawlerProcess
from scrapy import Request
from scrapy.spiders import Spider
from datetime import datetime
import time

In [ ]:
from scrapy.utils.project import get_project_settings

settings = get_project_settings()

settings.set("COOKIES_ENABLED", True, priority="cmdline")
settings.set("ROBOTSTXT_OBEY", False, priority="cmdline")

In [ ]:
import os
from mongodb_client import MongoDBClient
mongo_uri = os.getenv("MONGO_URI")
db_name = os.getenv("DB_NAME")
collection = os.getenv("COLLECTION_NAME")

In [ ]:
client = MongoDBClient(mongo_uri, db_name, collection)

In [ ]:
class ElPaisSpider(Spider):
    name = "elpais"
    allowed_domains = ["elpais.bo"]
    start_urls = [
        f"https://elpais.bo/tags/view/Feminicidio?page={i}" for i in range(1, 50)
    ]
    
    avoid_sections = ["/reportajes/", "/multimedia/", "/gobernacion-tarija/", "/opinion/", "/alcaldia-tarija/","/economia/", "/internacional/", "//"]
    
    user_agents = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 Safari/537.36 Edg/134.0.0.0",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:120.0) Gecko/20100101 Firefox/120.0"
    ]
    
    custom_headers = {
                    "User-Agent": user_agents[1],
                    "Connection": 'keep_alive',
                    "Host": "elpais.bo",
                    "Accept-Encoding": "text/html",
                }

    def __init__(self):
        self.items = []
        self.mongo_client = client
    
    def date_formatter(self, url, date_format="%Y%m%d"):
        try:
            url_split = url.split("/")
            date_str = url_split[4][:8]
            date_publish = datetime.strptime(date_str, date_format)
            return date_publish
        except Exception as e:
            self.logger.error(f"Error al formatear fecha: {e}")
            return None
    
    def tittle_formatter(self, title):
        try:
            new_title = title.replace("“", '"').replace("”", '"').strip()
            return new_title
        except Exception as e:
            self.logger.error(f"Error al formatear título: {e}")
            return title
    
    def tag_formatter(self, tags):
        try:
            list_tags = [t.lower().replace("#", "") for t in tags]
            return list_tags
        except Exception as e:
            self.logger.error(f"Error al formatear tags: {e}")
            return tags
    
    def section_formatter(self, url):
        try:
            url_split = url.split("/")
            section = url_split[3]
            return section
        except Exception as e:
            self.logger.error(f"Error al formatear sección: {e}")
            return url

    def body_formatter(self, body):
        try:
            new_body = [
                b.strip()
                .replace("\xa0", " ")
                .replace('\"', "")
                .replace("\ufeff", " ")
                .replace("“", '"')
                .replace("”", '"')
                .replace("\u200b", " ")
                for b in body
            ]
            new_body = [b for b in new_body if b != " "]
            return new_body
        except Exception as e:
            self.logger.error(f"Error al formatear cuerpo: {e}")
            return body

    def start_requests(self):
        for url in self.start_urls:
            self.logger.info(f"Enviando request a: {url}")
            yield Request(url=url, callback=self.parse_response, headers=self.custom_headers)
            
    
    def parse_response(self, response):
        self.logger.info(f"Recibida respuesta: {response.url}")
        try:
            noticias = response.xpath("(//ul[contains(@class,'uk-switcher')]//li)[1]//a[@class='link-news']/@href").getall()
            
            self.logger.info(f"Total noticias encontradas: {len(noticias)}")
            for noticia in noticias:
                if not any(s in noticia for s in self.avoid_sections):
                    self.logger.info(f"Enviando request a: {noticia}")
                    yield Request(url=noticia, callback=self.parse_news, headers=self.custom_headers)
        except Exception as e:
            self.logger.error(f"Error al procesar la respuesta JSON: {e}")
            return
    
    def parse_news(self, response):
        title = response.xpath('//h1[@class="ep_post_title"]/text()').get()
        item = {}
        item["url"] = response.url
        item["title"] = self.tittle_formatter(title)
        tags = response.xpath("//ul[@class='uk-subnav']//li//a/text()").getall()
        item["tags"] = self.tag_formatter(tags)
        item["section"] = self.section_formatter(response.url)
        body = response.xpath("//div[contains(@class,'note-body')]//p//text()").getall()
        item["body"] = self.body_formatter(body)
        item["date_published"] = self.date_formatter(response.url)
        item["source"] = self.name
        self.items.append(item)
        self.logger.info(f"Noticia agregada: {item['url']}")
        time.sleep(2)
    
    def close(self, reason):
        self.mongo_client.connect()
        self.logger.info("Guardando datos en MongoDB")
        for item in self.items:
            try:
                self.mongo_client.insert_new_document(item, "url")
            except Exception as e:
                self.logger.error(f"Error al insertar en MongoDB: {e}")
        self.mongo_client.close()
        self.logger.info(f"Spider cerrado por la razón: {reason}")

In [ ]:
process = CrawlerProcess()
process.crawl(ElPaisSpider)
process.start(settings)